# Weeks 4-5: California and Flagged Row Investigation

This notebook investigates the first-stage cleaned sold dataset before final cleaning decisions are applied. It reviews state issues and every data-quality flag.

In [109]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PRE_FILTER_FILE = PROCESSED_DIR / "crmls_sold_cleaned_before_ca_filter_202401_202606.csv"
pre_filter = pd.read_csv(PRE_FILTER_FILE, low_memory=False)

print("Investigation dataset shape:", pre_filter.shape)

Investigation dataset shape: (447991, 67)


## 1. Investigate State Values

Visualize the state distribution and inspect records that are not explicitly identified as California.

In [110]:
state = pre_filter["StateOrProvince"].astype("string").str.strip().str.upper()
state_counts = state.fillna("MISSING").value_counts().rename_axis("state").reset_index(name="row_count")
display(state_counts)

non_ca_mask = state.notna() & ~state.isin(["CA", "CALIFORNIA"])
missing_state_mask = state.isna() | state.eq("")

print("Explicit non-California rows:", int(non_ca_mask.sum()))
print("Missing-state rows:", int(missing_state_mask.sum()))

,state,row_count
0,CA,447961
1,AZ,10
2,OS,4
3,NV,2
4,CO,2
5,FL,2
6,ID,1
7,TX,1
8,GA,1
9,ME,1


Explicit non-California rows: 29
Missing-state rows: 1


In [111]:
display(pre_filter.loc[missing_state_mask])

,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,CloseDate,ClosePrice,Latitude,Longitude,UnparsedAddress,...,negative_timeline_flag,missing_coordinate_flag,zero_coordinate_flag,positive_longitude_flag,implausible_coordinate_flag,price_per_sqft,close_price_outlier_flag,living_area_outlier_flag,price_per_sqft_outlier_flag,possible_non_standard_residential_flag
434753,Wood,False,False,550000.0,1159827154,2026-06-09,550000.0,34.006183,-118.174188,2329 Connor Avenue,...,False,False,False,False,False,617.977528,False,False,False,False


In [112]:
# Is the missing-state data in California? Let's check the lat/lon bounding box for California.
def is_in_california_bbox(lat, lon):
    """Broad California bounding box check (same logic as week4_5_clean_sold_data.py)."""
    return (32 <= lat <= 42) and (-125 <= lon <= -114)

lat, lon = 34.006183, -118.174188
print(is_in_california_bbox(lat, lon))  # True


True


In [113]:
# Which city is it in
import requests
import time

def get_city(lat, lon):
    """Reverse-geocode a lat/long to city name using OpenStreetMap Nominatim."""
    url = "https://nominatim.openstreetmap.org/reverse"
    params = {"lat": lat, "lon": lon, "format": "json"}
    headers = {"User-Agent": "real-estate-dashboard-project"}  # Nominatim requires a UA

    response = requests.get(url, params=params, headers=headers, timeout=10)
    response.raise_for_status()
    address = response.json().get("address", {})

    return address.get("city") or address.get("town") or address.get("village")

lat, lon = 34.006183, -118.174188
print(get_city(lat, lon))  # Monterey Park


Commerce


In [114]:
# Create the next cleaning-stage dataset after reviewing state values.
# Keep explicit California records, plus missing-state records whose
# coordinates fall inside the California bounding box. The single
# missing-state row was also manually reviewed above and confirmed as CA.
plausible_ca_coordinates = (
    pre_filter["Latitude"].between(32, 42, inclusive="both")
    & pre_filter["Longitude"].between(-125, -114, inclusive="both")
)
keep_ca = state.isin(["CA", "CALIFORNIA"]) | (
    missing_state_mask & plausible_ca_coordinates
)

clean_date = pre_filter.loc[keep_ca].copy()
CLEAN_DATE_FILE = PROCESSED_DIR / "crmls_sold_cleaned_analysis_ready_202401_202606.csv"
clean_date.to_csv(CLEAN_DATE_FILE, index=False)

print("Rows before state cleaning:", f"{len(pre_filter):,}")
print("Rows removed:", f"{int((~keep_ca).sum()):,}")
print("Rows in clean_date:", f"{len(clean_date):,}")
print("Saved:", CLEAN_DATE_FILE)

Rows before state cleaning: 447,991
Rows removed: 29
Rows in clean_date: 447,962
Saved: /Users/baixuezhang/Documents/IDX Exchange/real-estate-dashboard-project/data/processed/crmls_sold_cleaned_analysis_ready_202401_202606.csv


In [115]:
# Reload the saved analysis-ready CSV so every following investigation
# examines the file itself rather than the earlier in-memory dataframe.
analysis_ready = pd.read_csv(CLEAN_DATE_FILE, low_memory=False)
analysis_state = analysis_ready["StateOrProvince"].astype("string").str.strip().str.upper()

print("Analysis-ready shape:", analysis_ready.shape)
display(
    analysis_state.fillna("MISSING")
    .value_counts()
    .rename_axis("state")
    .reset_index(name="row_count")
)

Analysis-ready shape: (447962, 67)


,state,row_count
0,CA,447961
1,MISSING,1


## 2. Review Every Flagged Category

Review the quality flags already created by the cleaning script. A flag identifies a record for review and does not automatically mean that the transaction should be deleted.

In [116]:
print(analysis_ready.columns.tolist())

['Flooring', 'ViewYN', 'PoolPrivateYN', 'OriginalListPrice', 'ListingKey', 'CloseDate', 'ClosePrice', 'Latitude', 'Longitude', 'UnparsedAddress', 'LivingArea', 'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName', 'ListAgentFullName', 'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName', 'CountyOrParish', 'MlsStatus', 'ElementarySchool', 'AttachedGarageYN', 'ParkingTotal', 'PropertySubType', 'LotSizeAcres', 'YearBuilt', 'BathroomsTotalInteger', 'City', 'BuyerAgencyCompensation', 'BedroomsTotal', 'ContractStatusChangeDate', 'PurchaseContractDate', 'ListingContractDate', 'StateOrProvince', 'MiddleOrJuniorSchool', 'FireplaceYN', 'Stories', 'HighSchool', 'Levels', 'LotSizeArea', 'MainLevelBedrooms', 'NewConstructionYN', 'GarageSpaces', 'HighSchoolDistrict', 'PostalCode', 'AssociationFee', 'LotSizeSquareFeet', 'year_month', 'rate_30yr_fixed', 'invalid_close_price_flag', 'invalid_living_area_flag', 'negative_days_on_market_flag', 'negative_bedrooms_flag', 'negative_bath

In [ ]:
# Load the analysis-ready dataset for the flag investigation. The quality
# flags were computed and saved by the cleaning workflow; numeric and flag
# columns come back with correct dtypes automatically, so only the date
# fields (needed for timeline comparisons below) are converted here.
investigation = pd.read_csv(CLEAN_DATE_FILE, low_memory=False)

for field in [
    "CloseDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "ContractStatusChangeDate",
]:
    if field in investigation.columns:
        investigation[field] = pd.to_datetime(investigation[field], errors="coerce")

print("Investigation dataset shape:", investigation.shape)

In [117]:
# Summarize every quality flag: how many rows are flagged and what share
# of the analysis-ready dataset they represent.
investigation = pd.read_csv(CLEAN_DATE_FILE, low_memory=False)

flag_columns = sorted(
    column for column in investigation.columns if column.endswith("_flag")
)

flag_summary = pd.DataFrame({
    "flagged_column": flag_columns,
    "flagged_count": [
        int(investigation[column].fillna(False).astype(bool).sum())
        for column in flag_columns
    ],
})
flag_summary["flagged_pct"] = (
    flag_summary["flagged_count"] / len(investigation) * 100
).round(4)

flag_summary = flag_summary.sort_values(
    "flagged_count", ascending=False
).reset_index(drop=True)

display(flag_summary)


,flagged_column,flagged_count,flagged_pct
0,possible_non_standard_residential_flag,9798,2.1872
1,price_per_sqft_outlier_flag,4476,0.9992
2,living_area_outlier_flag,4472,0.9983
3,close_price_outlier_flag,4421,0.9869
4,missing_coordinate_flag,4375,0.9766
5,negative_timeline_flag,530,0.1183
6,purchase_after_close_flag,240,0.0536
7,invalid_living_area_flag,165,0.0368
8,implausible_coordinate_flag,85,0.0190
9,listing_after_close_flag,68,0.0152


In [118]:
empty_flags = [
    column for column in investigation.columns
    if column.endswith("_flag")
    and not investigation[column].fillna(False).astype(bool).any()
]
investigation = investigation.drop(columns=empty_flags)
print("Dropped empty flags:", empty_flags)


Dropped empty flags: ['negative_bedrooms_flag', 'negative_bathrooms_flag']


In [119]:
# Investigate the single invalid ClosePrice record, then remove it.
invalid_price_mask = investigation["ClosePrice"].le(0).fillna(False)

review_columns = [
    "ListingKey", "CloseDate", "ListingContractDate", "ClosePrice",
    "ListPrice", "LivingArea", "PropertySubType", "City", "CountyOrParish",
]
review_columns = [column for column in review_columns if column in investigation.columns]

print(f"Rows with invalid ClosePrice: {int(invalid_price_mask.sum())}")
print("ClosePrice values:", investigation.loc[invalid_price_mask, "ClosePrice"].tolist())
display(investigation.loc[invalid_price_mask, review_columns])

# Drop the invalid row: ClosePrice is essential to sold-market analysis,
# and a price of 0 is a data-entry gap, not a real sale price.
rows_before = len(investigation)
investigation = investigation.loc[~invalid_price_mask].copy()

print(f"\nRows before: {rows_before:,}")
print(f"Rows removed: {rows_before - len(investigation):,}")
print(f"Rows after: {len(investigation):,}")


Rows with invalid ClosePrice: 1
ClosePrice values: [0.0]


,ListingKey,CloseDate,ListingContractDate,ClosePrice,ListPrice,LivingArea,PropertySubType,City,CountyOrParish
273584,1117019091,2025-07-30,2025-06-11,0.0,1350000.0,2001.0,SingleFamilyResidence,Temple City,Los Angeles



Rows before: 447,962
Rows removed: 1
Rows after: 447,961


In [120]:
# --- Handle zero coordinates: placeholder values, treat as missing. ---
zero_mask = investigation["Latitude"].eq(0) | investigation["Longitude"].eq(0)
print(f"Zero-coordinate rows set to missing: {int(zero_mask.sum())}")
investigation.loc[zero_mask, ["Latitude", "Longitude"]] = pd.NA

# --- Handle positive longitude: recover dropped minus signs when the
# --- flipped coordinate lands inside California; otherwise treat as missing.
positive_mask = investigation["Longitude"].gt(0).fillna(False)
flippable = (
    positive_mask
    & investigation["Latitude"].between(32, 42)
    & (-investigation["Longitude"]).between(-125, -114)
)

# ListingKey 1084837618 (East Los Angeles) passes the bounding-box check
# after the sign flip, but its latitude (40) is far from Los Angeles (~34),
# so the coordinate is unreliable and is treated as missing instead.
bad_east_la = investigation["ListingKey"].eq(1084837618)
flippable = flippable & ~bad_east_la

print(f"Positive-longitude rows: {int(positive_mask.sum())}")
print(f"  recovered by sign flip: {int(flippable.sum())}")
print(f"  unrecoverable, set to missing: {int((positive_mask & ~flippable).sum())}")

investigation.loc[flippable, "Longitude"] = -investigation.loc[flippable, "Longitude"]
investigation.loc[positive_mask & ~flippable, ["Latitude", "Longitude"]] = pd.NA


Zero-coordinate rows set to missing: 37
Positive-longitude rows: 31
  recovered by sign flip: 26
  unrecoverable, set to missing: 5


In [121]:
# --- Investigate listing_after_close: 68 rows where the listing date falls
# --- after the close date. The gap is small (median 5 days, max 31 days),
# --- and in 65 of 68 rows the purchase contract predates the "listing" date,
# --- so these look like off-market sales entered into the MLS after closing:
# --- ListingContractDate reflects the data-entry date, not a real listing date.

# Check the flag column values first.
print("Unique values:", investigation["listing_after_close_flag"].unique())
display(
    investigation["listing_after_close_flag"]
    .value_counts(dropna=False)
    .rename_axis("value")
    .reset_index(name="row_count")
)

listing_after_close_mask = investigation["listing_after_close_flag"].fillna(False).astype(bool)

# Review the flagged rows before changing anything.
display(
    investigation.loc[
        listing_after_close_mask,
        ["ListingKey", "ListingContractDate", "PurchaseContractDate",
         "CloseDate", "DaysOnMarket", "ClosePrice", "City"],
    ].head(15)
)

# In 3 of the 68 rows the purchase date equals the fake listing date and also
# falls after close — signing a purchase contract after closing is impossible,
# so that purchase date cannot be real either. Identify them BEFORE clearing
# the listing dates.
fake_purchase_mask = (
    listing_after_close_mask
    & investigation["PurchaseContractDate"].notna()
    & investigation["CloseDate"].notna()
    & investigation["PurchaseContractDate"].gt(investigation["CloseDate"])
)

# The sales themselves are real (close date and price are valid), so retain
# all rows and set only the impossible date fields to missing.
investigation.loc[listing_after_close_mask, "ListingContractDate"] = pd.NaT
investigation.loc[fake_purchase_mask, "PurchaseContractDate"] = pd.NaT

print(f"ListingContractDate set to missing: {int(listing_after_close_mask.sum())} rows")
print(f"PurchaseContractDate also set to missing: {int(fake_purchase_mask.sum())} rows")


Unique values: [False  True]


,value,row_count
0,False,447893
1,True,68


,ListingKey,ListingContractDate,PurchaseContractDate,CloseDate,DaysOnMarket,ClosePrice,City
20,1059827487,2024-01-31,2024-01-15,2024-01-30,0,2160000.0,Rancho Santa Fe
81,1059357897,2024-01-26,2024-01-10,2024-01-25,0,2705000.0,Del Mar
710,1054052014,2024-01-02,2023-12-04,2024-01-01,0,1600000.0,Oceanside
24269,1065738283,2024-04-08,2024-03-27,2024-03-29,0,1075000.0,Newbury Park
24389,1063548623,2024-03-21,2024-03-03,2024-03-20,0,1650000.0,San Diego
75918,1076675910,2024-06-20,2024-06-03,2024-06-13,0,715000.0,San Diego
110404,1079658093,2024-08-14,2024-08-05,2024-08-13,0,5600000.0,La Jolla
129033,1080048344,2024-10-02,2024-09-03,2024-09-19,16,1305000.0,Los Angeles
129223,1079998553,2024-09-30,2024-08-20,2024-09-05,16,995000.0,Los Angeles
129857,1079658804,2024-09-28,2024-09-05,2024-09-24,20,1275000.0,Los Angeles


ListingContractDate set to missing: 68 rows
PurchaseContractDate also set to missing: 3 rows


In [123]:
# ============================================================
# Investigate implausible_coordinate_flag: are these bad
# coordinates on California homes, or homes outside California?

# 85 implausible_coordinate_flag, 31 fillped signs, 37 of zeros turned to missing. How many remain after those steps? 85-31-37=17
# ============================================================

# --- Step 1: list the remaining flagged rows (positive-longitude and
# --- zero-coordinate cases were already handled above). For each row,
# --- show the location fields side by side with the coordinates so
# --- they can be cross-checked against each other.
remaining_implausible = (
    investigation["Latitude"].notna()
    & investigation["Longitude"].notna()
    & (
        ~investigation["Latitude"].between(32, 42)
        | ~investigation["Longitude"].between(-125, -114)
    )
)
print(f"Remaining implausible-coordinate rows: {int(remaining_implausible.sum())}")
display(
    investigation.loc[
        remaining_implausible,
        ["ListingKey", "City", "CountyOrParish", "StateOrProvince",
         "PostalCode", "Latitude", "Longitude"],
    ]
)

Remaining implausible-coordinate rows: 17


,ListingKey,City,CountyOrParish,StateOrProvince,PostalCode,Latitude,Longitude
27959,1061262345,Sugarloaf,San Bernardino,CA,92386,48.428421,-123.365644
75159,1046655870,Carmel,Monterey,CA,93923,10.305220,-84.810691
88668,1065220693,Laguna Woods,Orange,CA,92637,-38.718318,-62.266348
91957,1048521271,Cobb,Lake,CA,95426,56.130366,-106.346771
116547,1077113292,Corona,Riverside,CA,92882,33.876203,-177.646696
161745,1089708914,Riverside,Riverside,CA,92503,-117.472493,-117.472493
221772,1093448041,Outside Area (Outside U.S.) Foreign Country,Foreign Country,CA,22880,31.853187,-116.613551
221775,1093445263,Outside Area (Outside Ca),Other State,CA,22870,31.868462,-116.645616
252998,1103320218,Palmdale,Los Angeles,CA,93550,31.512886,-118.096779
276291,1114694734,Monterey,Monterey,CA,92940,20.748250,-97.635825


In [124]:
# --- Step 2: the City/County values above are suspicious — some literally
# --- say "Outside Area" or "Foreign Country". Count them to see the pattern.
display(
    investigation.loc[remaining_implausible, "CountyOrParish"]
    .value_counts()
    .rename_axis("CountyOrParish")
    .reset_index(name="row_count")
)

,CountyOrParish,row_count
0,Los Angeles,3
1,Monterey,2
2,Riverside,2
3,Foreign Country,2
4,Other State,2
5,San Bernardino,1
6,Orange,1
7,Lake,1
8,San Joaquin,1
9,Other,1


In [125]:
# --- Step 3: verify with an independent field: the zip code. California
# --- zip codes fall in 90001-96162. A non-CA zip that AGREES with the
# --- out-of-state coordinate means the property is genuinely outside CA
# --- (zip 228xx here is Baja California, Mexico; 86314 is Prescott AZ).
zip_numeric = pd.to_numeric(
    investigation["PostalCode"].astype("string").str.slice(0, 5), errors="coerce"
)
ca_zip = zip_numeric.between(90001, 96162)
display(
    investigation.loc[
        remaining_implausible & ~ca_zip,
        ["ListingKey", "City", "CountyOrParish", "PostalCode",
         "Latitude", "Longitude"],
    ]
)

,ListingKey,City,CountyOrParish,PostalCode,Latitude,Longitude
221772,1093448041,Outside Area (Outside U.S.) Foreign Country,Foreign Country,22880,31.853187,-116.613551
221775,1093445263,Outside Area (Outside Ca),Other State,22870,31.868462,-116.645616
319242,1134323077,Palmdale,Los Angeles,83551,34.619473,-11.230259
326006,1123830005,Other,Other,86314,34.559557,-112.346328
359334,1099893565,Other,Foreign Country,22785,31.786721,-116.608868
359361,1092500840,Other,Other State,22880,31.853002,-116.614420


In [126]:
# Remove the rows confirmed to be outside California: zip code and
# coordinates agree on the same non-CA location (4 Ensenada MX,
# 1 Prescott AZ, 1 Las Vegas NV). The uncertain Palmdale row is kept.
truly_outside_ca_keys = [
    1093448041,  # Ensenada, Mexico
    1093445263,  # Ensenada, Mexico
    1099893565,  # Ensenada, Mexico
    1092500840,  # Ensenada, Mexico
    1123830005,  # Prescott Valley, AZ
]
delete_mask = investigation["ListingKey"].isin(truly_outside_ca_keys)
rows_before = len(investigation)
investigation = investigation.loc[~delete_mask].copy()
print(f"Removed {rows_before - len(investigation)} non-CA rows: {rows_before:,} -> {len(investigation):,}")


Removed 5 non-CA rows: 447,961 -> 447,956


In [127]:
# --- Step 4: if some rows have out-of-CA labels, similar rows may exist
# --- elsewhere in the dataset with coordinates that happen to pass the
# --- loose bounding box (it covers slivers of NV and AZ). Sweep the whole
# --- dataset for the same suspicious City/County labels and non-CA zips.
suspicious_labels = (
    investigation["CountyOrParish"].isin(["Foreign Country", "Other State"])
    | investigation["City"].astype("string").str.contains(
        "Outside Ca|Outside U.S", na=False, regex=True
    )
)
display(
    investigation.loc[
        suspicious_labels | (investigation["Latitude"].notna() & ~ca_zip),
        ["ListingKey", "City", "CountyOrParish", "StateOrProvince",
         "PostalCode", "Latitude", "Longitude"],
    ]
)
# The sweep surfaces one more row the bounding box missed: zip 89139 with
# coordinates in Las Vegas, NV — the bbox includes a sliver of Nevada.

,ListingKey,City,CountyOrParish,StateOrProvince,PostalCode,Latitude,Longitude
17893,1054001583,Laguna Niguel,Orange,CA,22677,33.567560,-117.705078
29845,1060210244,Garden Grove,Orange,CA,98245,33.782176,-118.037402
39497,1043828713,Valencia,Los Angeles,CA,97381,34.422340,-118.609145
88937,1065131270,Moreno Valley,Riverside,CA,82553,33.895254,-117.212419
99713,1076021376,Chico,Butte,CA,05073,39.791299,-121.889783
120728,1076453744,Duarte,Los Angeles,CA,01010,36.778261,-119.417932
190086,1095741747,Canyon Country,Los Angeles,CA,19351,34.421762,-118.479670
217220,1103138741,Long Beach,Los Angeles,CA,980803,33.766919,-118.172263
222998,1082254223,Fontana,San Bernardino,CA,02336,34.172285,-117.458458
246083,1109528025,Outside Area (Outside Ca),Nevada,CA,89139,36.052659,-115.222332


In [128]:
# The sweep surfaces TWO non-CA properties whose coordinates sit inside the
# loose bounding box: Las Vegas and Henderson, NV. City, county, zip, and
# coordinates all agree on Nevada for both. All other swept rows are CA
# homes with corrupted zip codes (mostly a wrong first digit of 9xxxx),
# confirmed by consistent CA city/county/coordinates — they are kept.
nevada_keys = [
    1109528025,  # Las Vegas, NV — zip 89139, coords (36.05, -115.22)
    1146057687,  # Henderson, NV — County Clark, zip 89011, coords (36.08, -115.02)
]
nevada_mask = investigation["ListingKey"].isin(nevada_keys)
rows_before = len(investigation)
investigation = investigation.loc[~nevada_mask].copy()
print(f"Removed {rows_before - len(investigation)} Nevada rows: "
      f"{rows_before:,} -> {len(investigation):,}")


Removed 2 Nevada rows: 447,956 -> 447,954


### Zip Code Typos, Not Out-of-State Properties

Apart from the two Nevada rows removed above, every row surfaced by the
non-CA-zip sweep is a genuine California transaction with a corrupted zip
code. Three independent fields — City, CountyOrParish, and the coordinates —
all agree on a California location for each row; the zip code is the only
field that disagrees, so the zip is the error.

The corruption follows a clear pattern: most values are a valid California
zip (9xxxx) with a mistyped first digit:

| City | Recorded zip | Actual zip |
| --- | --- | --- |
| Laguna Niguel | 22677 | 92677 |
| Fontana | 02336 | 92336 |
| Palmdale | 83551 | 93551 |
| Templeton | 73465 | 93465 |
| Corning | 06021 | 96021 |
| Long Beach | 980803 | 90803 (extra digit) |
| Corona | 88888 | placeholder value |

The repeated values on Rialto (20448) and Ontario (20536) rows suggest a
systematic feed issue rather than one-off typing mistakes.

**Decision:** these rows are kept — the transactions are valid California
sales, and their zip codes are repaired in two ways. Zip codes whose error
maps unambiguously to a real zip of the row's city (a mistyped first digit,
transposed digits, or one extra digit) are corrected directly. Values with
no recoverable typo pattern (the repeated feed values 20448/20536/20205,
the 88888 placeholder, and the ambiguous 05073) are instead recovered by
reverse-geocoding the row's coordinates (OpenStreetMap Nominatim), which
were already verified to agree with the City and County fields. A geocoded
result is accepted only if it is a valid California zip (90001–96162);
otherwise the zip is set to missing rather than guessed.
 



In [129]:
import time
import requests

# ============================================================
# Repair the corrupted zip codes on the remaining CA rows.
# 1) Single-edit typos are corrected from the city's real zip.
# 2) Values with no typo pattern are recovered by reverse-
#    geocoding the row's verified coordinates (Nominatim).
# ============================================================

# --- Part 1: unambiguous single-edit typos ---
ZIP_FIXES = {
    "01010":  "91010",  # Duarte
    "80802":  "90802",  # Long Beach
    "980803": "90803",  # Long Beach (extra digit)
    "97381":  "91381",  # Valencia
    "96962":  "96062",  # Millville
    "83551":  "93551",  # Palmdale
    "83536":  "93536",  # Lancaster
    "82630":  "92630",  # Lake Forest
    "82553":  "92553",  # Moreno Valley
    "73465":  "93465",  # Templeton
    "49603":  "94603",  # Oakland (transposed 49 -> 94)
    "39534":  "93534",  # Lancaster (transposed 39 -> 93)
    "22677":  "92677",  # Laguna Niguel
    "19351":  "91351",  # Canyon Country (transposed 19 -> 91)
    "06021":  "96021",  # Corning
    "02637":  "92637",  # Laguna Woods
    "02336":  "92336",  # Fontana
    "98245":  "92845",  # Garden Grove (transposed 8 <-> 2)
}
# No recoverable typo pattern -> recover from coordinates instead.
ZIP_FROM_COORDS = ["20448", "20536", "20205", "88888", "05073"]

zips = investigation["PostalCode"].astype("string").str.strip()
fixable_mask = zips.isin(ZIP_FIXES)
geocode_mask = (
    zips.isin(ZIP_FROM_COORDS)
    & investigation["Latitude"].notna()
    & investigation["Longitude"].notna()
)

investigation.loc[fixable_mask, "PostalCode"] = zips[fixable_mask].map(ZIP_FIXES)
print(f"Zip typos corrected: {int(fixable_mask.sum())}")

# --- Part 2: recover the rest by reverse-geocoding the coordinates ---
def reverse_geocode_zip(lat, lon):
    """Look up the zip code for a coordinate via OpenStreetMap Nominatim."""
    url = "https://nominatim.openstreetmap.org/reverse"
    params = {"lat": lat, "lon": lon, "format": "json"}
    headers = {"User-Agent": "real-estate-dashboard-project"}
    response = requests.get(url, params=params, headers=headers, timeout=10)
    response.raise_for_status()
    return response.json().get("address", {}).get("postcode")

coordinate_cache = {}  # identical coordinates -> one API call
for idx in investigation.index[geocode_mask]:
    lat = investigation.at[idx, "Latitude"]
    lon = investigation.at[idx, "Longitude"]
    key = (round(lat, 5), round(lon, 5))
    if key not in coordinate_cache:
        coordinate_cache[key] = reverse_geocode_zip(lat, lon)
        time.sleep(1)  # Nominatim rate limit: 1 request/second
    new_zip = coordinate_cache[key]

    # Accept only a valid California zip; otherwise leave the zip missing.
    if new_zip and new_zip[:5].isdigit() and 90001 <= int(new_zip[:5]) <= 96162:
        investigation.at[idx, "PostalCode"] = new_zip[:5]
    else:
        investigation.at[idx, "PostalCode"] = pd.NA

recovered = int(investigation.loc[geocode_mask, "PostalCode"].notna().sum())
print(f"Zips recovered from coordinates: {recovered} of {int(geocode_mask.sum())}")

# --- Review the repaired rows ---
display(
    investigation.loc[
        fixable_mask | geocode_mask,
        ["ListingKey", "City", "CountyOrParish", "PostalCode", "Latitude", "Longitude"],
    ].sort_values("City")
)

# --- Verify: every zip in the dataset is now a valid CA zip or missing ---
check = pd.to_numeric(
    investigation["PostalCode"].astype("string").str.slice(0, 5), errors="coerce"
)
non_ca_left = int((~check.between(90001, 96162) & check.notna()).sum())
print(f"Non-CA zips remaining in dataset: {non_ca_left}")


Zip typos corrected: 18
Zips recovered from coordinates: 17 of 17


,ListingKey,City,CountyOrParish,PostalCode,Latitude,Longitude
190086,1095741747,Canyon Country,Los Angeles,91351,34.421762,-118.479670
99713,1076021376,Chico,Butte,95973,39.791299,-121.889783
413039,1138489256,Corning,Tehama,96021,39.896130,-122.101019
277370,1114499712,Corona,Other,92880,33.953980,-117.562435
120728,1076453744,Duarte,Los Angeles,91010,36.778261,-119.417932
222998,1082254223,Fontana,San Bernardino,92336,34.172285,-117.458458
29845,1060210244,Garden Grove,Orange,92845,33.782176,-118.037402
17893,1054001583,Laguna Niguel,Orange,92677,33.567560,-117.705078
249167,1108647034,Lake Forest,Orange,92630,33.657835,-117.672831
409869,1151531590,Lancaster,Los Angeles,93534,34.694671,-118.163127


Non-CA zips remaining in dataset: 0


In [130]:
# Investigate negative DaysOnMarket: the listing/purchase/close dates are
# internally consistent, so only the DOM field itself is corrupted (an
# artifact of the MLS source system's DOM accumulation logic).
negative_dom_mask = investigation["DaysOnMarket"].lt(0).fillna(False)

print(f"Negative DOM rows: {int(negative_dom_mask.sum())}")
display(
    investigation.loc[
        negative_dom_mask,
        ["ListingKey", "ListingContractDate", "PurchaseContractDate",
         "CloseDate", "DaysOnMarket", "City"],
    ].head(15)
)

# Retain the transactions and replace the invalid DOM with missing,
# so DOM-based metrics ignore these rows automatically.
investigation.loc[negative_dom_mask, "DaysOnMarket"] = pd.NA
print(f"DaysOnMarket set to missing for {int(negative_dom_mask.sum())} rows")


Negative DOM rows: 51


,ListingKey,ListingContractDate,PurchaseContractDate,CloseDate,DaysOnMarket,City
4,1063453216,2023-11-17,2023-12-19,2024-01-22,-36,Fort Bragg
393,1054197723,2024-01-08,2024-01-19,2024-01-19,-1,La Quinta
11204,1062122119,2024-02-24,2024-02-24,2024-02-26,-13,Indian Wells
11371,1061302766,2024-02-16,2024-02-16,2024-02-23,-10,Riverside
18688,1052574395,2023-12-12,2023-12-23,2024-02-05,-10,Oxnard
22995,1045653374,2023-09-25,2024-01-29,2024-02-29,-13,Beaumont
24387,1063549350,2024-01-30,2024-02-02,2024-03-21,-48,Ojai
24395,1063528331,2024-01-11,2024-01-22,2024-03-19,-58,Fort Bragg
24431,1063458939,2024-03-03,2024-03-04,2024-03-15,-14,Ventura
29415,1060256290,2024-02-07,2024-02-10,2024-03-08,-2,Camarillo


DaysOnMarket set to missing for 51 rows


In [141]:
# Verify the negative-DOM fix: the flag column still remembers which rows
# were affected, so use it to check their DaysOnMarket is now missing.
print("Rows with negative DOM now:", int(investigation["DaysOnMarket"].lt(0).sum()))

flagged_dom = investigation["negative_days_on_market_flag"].fillna(False).astype(bool)
print("Originally flagged rows:", int(flagged_dom.sum()))
print("Of those, DOM is now missing:", int(investigation.loc[flagged_dom, "DaysOnMarket"].isna().sum()))

display(
    investigation.loc[
        flagged_dom,
        ["ListingKey", "ListingContractDate", "PurchaseContractDate",
         "CloseDate", "DaysOnMarket", "City"],
    ].head(15)
)


Rows with negative DOM now: 0
Originally flagged rows: 51
Of those, DOM is now missing: 51


,ListingKey,ListingContractDate,PurchaseContractDate,CloseDate,DaysOnMarket,City
4,1063453216,2023-11-17,2023-12-19,2024-01-22,NaN,Fort Bragg
393,1054197723,2024-01-08,2024-01-19,2024-01-19,NaN,La Quinta
11204,1062122119,2024-02-24,2024-02-24,2024-02-26,NaN,Indian Wells
11371,1061302766,2024-02-16,2024-02-16,2024-02-23,NaN,Riverside
18688,1052574395,2023-12-12,2023-12-23,2024-02-05,NaN,Oxnard
22995,1045653374,2023-09-25,2024-01-29,2024-02-29,NaN,Beaumont
24387,1063549350,2024-01-30,2024-02-02,2024-03-21,NaN,Ojai
24395,1063528331,2024-01-11,2024-01-22,2024-03-19,NaN,Fort Bragg
24431,1063458939,2024-03-03,2024-03-04,2024-03-15,NaN,Ventura
29415,1060256290,2024-02-07,2024-02-10,2024-03-08,NaN,Camarillo


In [142]:
# --- Handle invalid_living_area_flag: 165 rows where LivingArea is exactly 0
# --- (no negatives). Zero is a "not recorded" placeholder, not a real size —
# --- 135 are single-family homes and 22 mobile homes that simply lack the
# --- field. Retain the transactions and set the invalid area to missing, so
# --- size- and price-per-sqft metrics ignore these rows automatically.
invalid_area_mask = investigation["LivingArea"].le(0).fillna(False)

print(f"Rows with LivingArea <= 0: {int(invalid_area_mask.sum())}")
display(
    investigation.loc[
        invalid_area_mask,
        ["ListingKey", "ClosePrice", "LivingArea", "PropertySubType", "City"],
    ].head(10)
)

investigation.loc[invalid_area_mask, "LivingArea"] = pd.NA
print(f"LivingArea set to missing for {int(invalid_area_mask.sum())} rows")


Rows with LivingArea <= 0: 165


,ListingKey,ClosePrice,LivingArea,PropertySubType,City
7701,1046380894,8000000.0,0.0,SingleFamilyResidence,Beverly Hills
21089,1046973939,9000000.0,0.0,SingleFamilyResidence,Malibu
22958,1045698780,1930000.0,0.0,SingleFamilyResidence,Los Angeles
22959,1045698270,1850000.0,0.0,SingleFamilyResidence,Los Angeles
35332,1054697515,1730000.0,0.0,SingleFamilyResidence,Santa Barbara
41048,1065199889,654000.0,0.0,SingleFamilyResidence,Concord
48834,1061828264,4950000.0,0.0,SingleFamilyResidence,Calabasas
50364,1061565144,1729000.0,0.0,SingleFamilyResidence,Los Angeles
52592,1060180994,2150000.0,0.0,SingleFamilyResidence,Los Angeles
52593,1060180989,2150000.0,0.0,SingleFamilyResidence,Los Angeles


LivingArea set to missing for 165 rows


In [143]:
# --- Handle purchase_after_close_flag: 240 rows where the purchase contract
# --- date falls AFTER the close date (median 3 days late, 75% within a week).
# --- Signing a purchase contract after escrow has closed is impossible, so
# --- the date is not a real signing date — same after-the-fact entry pattern
# --- as the fake listing dates handled above. The sales themselves are valid,
# --- so retain the rows and set only the impossible date to missing.
# --- (3 of the 240 were already cleared with the listing_after_close fix;
# --- selecting by the flag column makes this cell safe to re-run.)
purchase_after_close_mask = investigation["purchase_after_close_flag"].fillna(False).astype(bool)

print(f"Flagged rows: {int(purchase_after_close_mask.sum())}")
display(
    investigation.loc[
        purchase_after_close_mask,
        ["ListingKey", "ListingContractDate", "PurchaseContractDate",
         "CloseDate", "ClosePrice", "City"],
    ].head(10)
)

investigation.loc[purchase_after_close_mask, "PurchaseContractDate"] = pd.NaT
print(f"PurchaseContractDate set to missing for {int(purchase_after_close_mask.sum())} rows")


Flagged rows: 240


,ListingKey,ListingContractDate,PurchaseContractDate,CloseDate,ClosePrice,City
5,1061988701,2024-01-31,2024-03-04,2024-01-31,2340000.0,San Jose
9,1060105211,2024-01-31,2024-02-05,2024-01-31,2150000.0,Millbrae
10,1060066702,2024-01-29,2024-02-04,2024-01-29,2000000.0,San Mateo
18,1059861172,2024-01-29,2024-02-01,2024-01-29,1928800.0,Hayward
80,1059357902,2024-01-25,2024-01-26,2024-01-25,1100000.0,Santa Cruz
115,1058442145,2024-01-10,2024-01-21,2024-01-10,2950000.0,Watsonville
121,1058422805,2024-01-16,2024-01-19,2024-01-16,2000000.0,Burlingame
165,1058387951,2024-01-16,2024-01-18,2024-01-16,2155000.0,Fremont
246,1058155157,2024-01-13,2024-01-14,2024-01-13,899000.0,Rosemead
11196,1071117377,2024-02-21,2024-04-22,2024-02-21,1520000.0,Santa Clara


PurchaseContractDate set to missing for 240 rows


In [145]:
# Convert the date columns to real datetimes (they were loaded as strings).
# errors="coerce" also normalizes the NaT values set by the fixes above.
for field in [
    "CloseDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "ContractStatusChangeDate",
]:
    if field in investigation.columns:
        investigation[field] = pd.to_datetime(investigation[field], errors="coerce")

print(investigation[["CloseDate", "PurchaseContractDate", "ListingContractDate"]].dtypes)


CloseDate               datetime64[us]
PurchaseContractDate    datetime64[us]
ListingContractDate     datetime64[us]
dtype: object


In [146]:
# --- Investigate the last timeline issue: purchase date BEFORE listing date
# --- (the remaining piece of negative_timeline_flag). How far before, and
# --- does the gap size suggest different causes?
purchase_before_listing_mask = (
    investigation["PurchaseContractDate"].notna()
    & investigation["ListingContractDate"].notna()
    & investigation["PurchaseContractDate"].lt(investigation["ListingContractDate"])
)
print(f"Purchase-before-listing rows: {int(purchase_before_listing_mask.sum())}")

gap_days = (
    investigation.loc[purchase_before_listing_mask, "ListingContractDate"]
    - investigation.loc[purchase_before_listing_mask, "PurchaseContractDate"]
).dt.days

print("\nGap distribution (days purchase precedes listing):")
print(gap_days.describe())
print("\nGap buckets:")
print(pd.cut(gap_days, [0, 7, 30, 90, 365, 40000]).value_counts().sort_index())

# Look at the extremes first — gaps of years are likely year typos.
display(
    investigation.loc[purchase_before_listing_mask]
    .assign(gap_days=gap_days)
    .sort_values("gap_days", ascending=False)
    [["ListingKey", "ListingContractDate", "PurchaseContractDate",
      "CloseDate", "DaysOnMarket", "ClosePrice", "City", "gap_days"]]
    .head(20)
)


Purchase-before-listing rows: 225

Gap distribution (days purchase precedes listing):
count      225.000000
mean       178.884444
std       2426.526562
min          1.000000
25%          3.000000
50%         11.000000
75%         21.000000
max      36407.000000
dtype: float64

Gap buckets:
(0, 7]           96
(7, 30]         112
(30, 90]         14
(90, 365]         1
(365, 40000]      2
Name: count, dtype: int64


,ListingKey,ListingContractDate,PurchaseContractDate,CloseDate,DaysOnMarket,ClosePrice,City,gap_days
23630,1041330748,2023-07-21,1923-11-16,2024-02-28,88.0,785000.0,Millbrae,36407
270569,1119467517,2025-06-13,2023-06-13,2025-07-14,0.0,3000000.0,Los Angeles,731
196098,1079404126,2024-08-12,2024-01-10,2025-01-24,151.0,2215000.0,Carmel,215
397678,1157661435,2026-04-15,2026-02-03,2026-04-15,0.0,1515000.0,Newbury Park,71
369696,1153186301,2026-02-18,2025-12-19,2026-02-18,0.0,1625000.0,Thousand Oaks,61
141908,1089769265,2024-10-07,2024-08-19,2024-10-08,0.0,510000.0,San Diego,49
58669,1073157675,2024-04-30,2024-03-19,2024-05-07,0.0,1100000.0,Fallbrook,42
369841,1152629164,2026-02-01,2025-12-26,2026-02-04,0.0,735000.0,Valley Center,37
188074,1099893428,2025-01-26,2024-12-20,2025-01-28,7.0,730000.0,Long Beach,37
219666,1100854269,2025-02-13,2025-01-07,2025-03-10,9.0,1100000.0,Los Angeles,37


In [147]:
# --- Resolve purchase-before-listing (225 rows), two distinct causes. ---

# Cause 1: three year typos in PurchaseContractDate. Each correction is
# verified: the repaired timeline becomes fully consistent, and for the
# Carmel row the listing-to-purchase span exactly matches its DOM of 151.
PURCHASE_DATE_FIXES = {
    1041330748: "2023-11-16",  # Millbrae: 1923 -> 2023 (century typo)
    1119467517: "2025-06-13",  # Los Angeles: 2023 -> 2025 (matches listing date)
    1079404126: "2025-01-10",  # Carmel: 2024 -> 2025 (validated by DOM = 151)
}
for key, fixed_date in PURCHASE_DATE_FIXES.items():
    row_mask = investigation["ListingKey"].eq(key)
    investigation.loc[row_mask, "PurchaseContractDate"] = pd.Timestamp(fixed_date)
print(f"Purchase-date year typos fixed: {len(PURCHASE_DATE_FIXES)}")

# Cause 2: the remaining ~222 rows follow the familiar after-the-fact entry
# pattern — purchase -> close is valid, but the "listing" date sits after the
# purchase (often equal to the close date) with DOM 0: it is the MLS entry
# date, not a real listing date. Same treatment as listing_after_close:
# keep the rows, set the unreliable listing date to missing.
purchase_before_listing_mask = (
    investigation["PurchaseContractDate"].notna()
    & investigation["ListingContractDate"].notna()
    & investigation["PurchaseContractDate"].lt(investigation["ListingContractDate"])
)
print(f"Backfilled-entry rows: {int(purchase_before_listing_mask.sum())}")
investigation.loc[purchase_before_listing_mask, "ListingContractDate"] = pd.NaT
print(f"ListingContractDate set to missing for {int(purchase_before_listing_mask.sum())} rows")

# --- Verify: no timeline violations remain anywhere in the dataset. ---
l, p, c = (investigation["ListingContractDate"],
           investigation["PurchaseContractDate"],
           investigation["CloseDate"])
print("listing > close   :", int((l.notna() & c.notna() & (l > c)).sum()))
print("purchase > close  :", int((p.notna() & c.notna() & (p > c)).sum()))
print("purchase < listing:", int((p.notna() & l.notna() & (p < l)).sum()))


Purchase-date year typos fixed: 3
Backfilled-entry rows: 222
ListingContractDate set to missing for 222 rows
listing > close   : 0
purchase > close  : 0
purchase < listing: 0


In [140]:
# Check the unique values of the flag column
print(investigation["possible_non_standard_residential_flag"].unique())

# With counts (including NaN if any)
display(
    investigation["possible_non_standard_residential_flag"]
    .value_counts(dropna=False)
    .rename_axis("value")
    .reset_index(name="row_count")
)


[False  True]


,value,row_count
0,False,438156
1,True,9798


In [132]:
print(
    "Flag analysis: The three outlier flags (close_price_outlier_flag, "
    "living_area_outlier_flag, price_per_sqft_outlier_flag) each capture the "
    "top ~1% of records by design, since thresholds are set at the 99th "
    "percentile (ClosePrice > ~$5.6M, LivingArea > ~5,288 sqft, "
    "price_per_sqft > ~$1,895). They overlap heavily — the same luxury or "
    "multi-unit properties often trigger all three — and combine into "
    "possible_non_standard_residential_flag (9,798 rows, 2.19%). These are "
    "flagged for review, not removed: most are valid luxury sales, though a "
    "few extreme values suggest data-entry errors. The missing_coordinate_flag "
    "(4,375 rows, 0.98%) is unrelated to price: these are ordinary "
    "transactions lacking latitude/longitude, which are excluded from map "
    "visuals but remain valid for all other analysis."
)


Flag analysis: The three outlier flags (close_price_outlier_flag, living_area_outlier_flag, price_per_sqft_outlier_flag) each capture the top ~1% of records by design, since thresholds are set at the 99th percentile (ClosePrice > ~$5.6M, LivingArea > ~5,288 sqft, price_per_sqft > ~$1,895). They overlap heavily — the same luxury or multi-unit properties often trigger all three — and combine into possible_non_standard_residential_flag (9,798 rows, 2.19%). These are flagged for review, not removed: most are valid luxury sales, though a few extreme values suggest data-entry errors. The missing_coordinate_flag (4,375 rows, 0.98%) is unrelated to price: these are ordinary transactions lacking latitude/longitude, which are excluded from map visuals but remain valid for all other analysis.


In [133]:
# Profile the 9,798 flagged rows to understand what they are.
flagged = investigation.loc[investigation["possible_non_standard_residential_flag"]]
normal = investigation.loc[~investigation["possible_non_standard_residential_flag"]]

print(f"Flagged rows: {len(flagged):,} ({len(flagged) / len(investigation) * 100:.2f}%)")

# 1. Compare price and size: flagged vs normal records.
comparison = pd.DataFrame({
    "flagged_median": [
        flagged["ClosePrice"].median(),
        flagged["LivingArea"].median(),
        flagged["price_per_sqft"].median(),
    ],
    "normal_median": [
        normal["ClosePrice"].median(),
        normal["LivingArea"].median(),
        normal["price_per_sqft"].median(),
    ],
}, index=["ClosePrice", "LivingArea", "price_per_sqft"])
comparison["flagged_vs_normal"] = (
    comparison["flagged_median"] / comparison["normal_median"]
).round(1).astype(str) + "x"
display(comparison)

# 2. What property subtypes are these? Apartment/multi-unit types stand out.
display(
    flagged["PropertySubType"]
    .value_counts(dropna=False)
    .rename_axis("PropertySubType")
    .reset_index(name="row_count")
    .assign(pct=lambda t: (t["row_count"] / len(flagged) * 100).round(2))
)

# 3. Where are they concentrated? Luxury markets should dominate.
display(
    flagged["City"]
    .value_counts()
    .head(15)
    .rename_axis("City")
    .reset_index(name="row_count")
)

# 4. Written conclusion, computed from the data itself.
multi_unit_types = ["Apartment", "Duplex", "Triplex", "Quadruplex", "MultiFamily"]
multi_unit_count = int(
    flagged["PropertySubType"].isin(multi_unit_types).sum()
)
median_price_ratio = flagged["ClosePrice"].median() / normal["ClosePrice"].median()

print(
    f"\nConclusion: the {len(flagged):,} flagged rows are predominantly "
    f"high-value transactions — their median close price is "
    f"{median_price_ratio:.1f}x that of normal records. "
    f"{multi_unit_count:,} of them are multi-unit types "
    f"(apartment/duplex/triplex/quadruplex), and the rest are mostly "
    f"luxury single-family homes concentrated in high-end markets. "
    f"They are retained with the flag so dashboards can include or "
    f"exclude them as needed."
)


Flagged rows: 9,798 (2.19%)


,flagged_median,normal_median,flagged_vs_normal
ClosePrice,5.100000e+06,810000.000000,6.3x
LivingArea,4.800000e+03,1633.000000,2.9x
price_per_sqft,1.720107e+03,531.674208,3.2x


,PropertySubType,row_count,pct
0,SingleFamilyResidence,9104,92.92
1,Condominium,483,4.93
2,Duplex,75,0.77
3,Townhouse,55,0.56
4,BoatSlip,18,0.18
5,NaN,17,0.17
6,MixedUse,15,0.15
7,Triplex,11,0.11
8,Quadruplex,9,0.09
9,ManufacturedOnLand,6,0.06


,City,row_count
0,Los Angeles,624
1,Newport Beach,525
2,Palo Alto,409
3,Beverly Hills,303
4,Los Altos,266
5,Laguna Beach,247
6,Manhattan Beach,237
7,Santa Monica,225
8,San Diego,224
9,Rancho Santa Fe,219



Conclusion: the 9,798 flagged rows are predominantly high-value transactions — their median close price is 6.3x that of normal records. 95 of them are multi-unit types (apartment/duplex/triplex/quadruplex), and the rest are mostly luxury single-family homes concentrated in high-end markets. They are retained with the flag so dashboards can include or exclude them as needed.


In [134]:
flag_columns = sorted(column for column in investigation.columns if column.endswith("_flag"))

print(f"Number of flagged columns: {len(flag_columns)}")

flag_summary = pd.DataFrame({
    "flag": flag_columns,
    "flagged_count": [int(investigation[column].fillna(False).astype(bool).sum()) for column in flag_columns],
})
flag_summary["flagged_pct"] = (flag_summary["flagged_count"] / len(investigation) * 100).round(4)
display(flag_summary.sort_values("flagged_count", ascending=False))

Number of flagged columns: 14


,flag,flagged_count,flagged_pct
10,possible_non_standard_residential_flag,9798,2.1873
11,price_per_sqft_outlier_flag,4476,0.9992
5,living_area_outlier_flag,4472,0.9983
0,close_price_outlier_flag,4421,0.9869
6,missing_coordinate_flag,4375,0.9767
8,negative_timeline_flag,530,0.1183
12,purchase_after_close_flag,240,0.0536
3,invalid_living_area_flag,165,0.0368
1,implausible_coordinate_flag,80,0.0179
4,listing_after_close_flag,68,0.0152


In [148]:
# --- Cleaning decision log: every flag investigated and resolved. ---

handled = pd.DataFrame([
    {"flag": "invalid_close_price_flag",     "rows": 1,
     "action": "row deleted (ClosePrice=0 was a data-entry gap)"},
    {"flag": "zero_coordinate_flag",         "rows": 37,
     "action": "coordinates set to missing (placeholder 0,0)"},
    {"flag": "positive_longitude_flag",      "rows": 31,
     "action": "26 recovered by sign flip, 5 set to missing"},
    {"flag": "implausible_coordinate_flag",  "rows": 85,
     "action": "overlaps zero/positive-longitude fixes; 6 non-CA rows deleted "
               "(4 Ensenada MX, Prescott AZ, Las Vegas NV via sweep), "
               "1 more NV row (Henderson) deleted; garbage coords left flagged"},
    {"flag": "corrupted zip codes",          "rows": 35,
     "action": "18 single-edit typos corrected; 17 recovered by reverse-geocoding "
               "verified coordinates (Nominatim)"},
    {"flag": "negative_days_on_market_flag", "rows": 51,
     "action": "DOM set to missing (dates consistent, DOM counter corrupted)"},
    {"flag": "listing_after_close_flag",     "rows": 68,
     "action": "listing date set to missing (entry date, not real listing); "
               "3 purchase dates also cleared"},
    {"flag": "purchase_after_close_flag",    "rows": 240,
     "action": "purchase date set to missing (impossible: signed after closing)"},
    {"flag": "purchase-before-listing (negative_timeline)", "rows": 225,
     "action": "3 year typos in purchase date fixed (timeline-validated); "
               "222 backfilled entries: listing date set to missing"},
    {"flag": "invalid_living_area_flag",     "rows": 165,
     "action": "LivingArea=0 set to missing (placeholder; incl. luxury homes)"},
    {"flag": "empty flags (bed/bath negative)", "rows": 0,
     "action": "columns dropped (zero hits)"},
])

no_action = pd.DataFrame([
    {"flag": "missing_coordinate_flag", "rows": 4375,
     "note": "already missing; excluded from map analysis only"},
    {"flag": "outlier flags + possible_non_standard_residential", "rows": 9798,
     "note": "flagged for review, kept by design (luxury/multi-unit sales)"},
])

print("=== RESOLVED ===")
display(handled)
print("=== KEPT AS FLAGGED (BY DESIGN) ===")
display(no_action)
print(f"\nFinal dataset: {len(investigation):,} rows x {investigation.shape[1]} columns")


=== RESOLVED ===


,flag,rows,action
0,invalid_close_price_flag,1,row deleted (ClosePrice=0 was a data-entry gap)
1,zero_coordinate_flag,37,"coordinates set to missing (placeholder 0,0)"
2,positive_longitude_flag,31,"26 recovered by sign flip, 5 set to missing"
3,implausible_coordinate_flag,85,overlaps zero/positive-longitude fixes; 6 non-...
4,corrupted zip codes,35,18 single-edit typos corrected; 17 recovered b...
5,negative_days_on_market_flag,51,"DOM set to missing (dates consistent, DOM coun..."
6,listing_after_close_flag,68,"listing date set to missing (entry date, not r..."
7,purchase_after_close_flag,240,purchase date set to missing (impossible: sign...
8,purchase-before-listing (negative_timeline),225,3 year typos in purchase date fixed (timeline-...
9,invalid_living_area_flag,165,LivingArea=0 set to missing (placeholder; incl...


=== KEPT AS FLAGGED (BY DESIGN) ===


,flag,rows,note
0,missing_coordinate_flag,4375,already missing; excluded from map analysis only
1,outlier flags + possible_non_standard_residential,9798,"flagged for review, kept by design (luxury/mul..."



Final dataset: 447,954 rows x 65 columns


In [ ]:
# --- Save the fully cleaned, analysis-ready dataset. ---
FINAL_FILE = PROCESSED_DIR / "crmls_sold_final_202401_202606.csv"
investigation.to_csv(FINAL_FILE, index=False)
print(f"Saved: {FINAL_FILE}")
print(f"Shape: {investigation.shape[0]:,} rows x {investigation.shape[1]} columns")